<a href="https://colab.research.google.com/github/ZarinRitu32/Myopia_ML/blob/main/SoftCom_01_Data_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SoftCom myopia-risk benchmark - Section 1 - Data ingest & validation

Replication and extension of Mu et al. 2024 (Front. Med. 11:1482788) on the provided **hybrid cohort (N=1500)**. Pipeline validation only - no clinical claims.

**Pipeline:** **SoftCom_01_Data_Validation** -> SoftCom_02_Preprocessing -> SoftCom_03_Benchmarking -> SoftCom_04_Explainable_AI -> SoftCom_05_Figures -> SoftCom_06_Export

All inputs and outputs live in Google Drive at `MyDrive/SoftCom_Myopia/`. Each notebook checks `artifacts/manifest.json` and stops if an upstream step has not run or its files have changed.

| | |
|---|---|
| **Reads** | `data/raw/full_myopia_hybrid_dataset_N1500.csv` (uploaded once) |
| **Writes** | `data/processed/cohort_validated.csv`, `tables/table0_validation_report.csv`, `tables/table1_characteristics.csv`, `artifacts/validation_report.json` |
| **Next** | `SoftCom_02_Preprocessing.ipynb` |

Runtime: Colab CPU is enough. Use *Runtime -> Run all*; accept the Drive prompt in the setup cell. For a quick trial run, set `"FAST_MODE": True` in the CONFIG cell of **every** notebook (fewer CV repeats/bootstraps); set it back to `False` for the reported results.

## 0. Install dependencies and set up Google Drive

In [ ]:
%pip install -q "shap>=0.47" xgboost lightgbm statsmodels imbalanced-learn

In [ ]:
"""Shared setup (identical in every SoftCom notebook).

Mounts Google Drive, defines CONFIG, the Drive folder layout, the run manifest
that links the six notebooks together, seeding, logging and figure helpers.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# ----------------------------------------------------------------------------- CONFIG
CONFIG: Dict[str, Any] = {
    "SEED": 42,
    "TEST_SIZE": 0.3,
    "N_SPLITS": 5,
    "N_REPEATS": 10,
    "N_BOOTSTRAP": 2000,
    "EXPECTED_N_ROWS": 1500,
    "DATA_FILENAME": "full_myopia_hybrid_dataset_N1500.csv",
    "PILOT_FILENAME": "pilot_real.csv",          # optional 73-row pilot; never merged
    "TARGET": "is_myopic",
    "SOURCE_LABEL": "hybrid",
    "COHORT_LABEL": "Hybrid cohort (N=1500) - pipeline validation only, not clinical evidence",
    "DRIVE_PROJECT_DIR": "SoftCom_Myopia",       # folder inside MyDrive holding everything
    "USE_SMOTE": False,                          # SMOTE inside pipeline only (imblearn)
    "USE_LASSO_SELECTION": True,                 # L1-logistic SelectFromModel inside pipeline
    "ABLATION": True,                            # also run FT-Transformer without attention
    "DEEP_MAX_EPOCHS": 200,
    "DEEP_PATIENCE": 20,
    "DEEP_BATCH_SIZE": 64,
    "PERM_N_REPEATS": 20,
    "SHAP_BACKGROUND": 100,
    "SHAP_N_EXPLAIN": 150,
    "SHAP_KERNEL_NSAMPLES": 200,
    "FAST_MODE": False,                          # True = quick trial run (same value in all notebooks)
}
CONFIG["FAST_MODE"] |= os.environ.get("SOFTCOM_FAST", "0") == "1"  # local smoke tests
if CONFIG["FAST_MODE"]:
    CONFIG.update(N_REPEATS=2, N_BOOTSTRAP=200, DEEP_MAX_EPOCHS=25, DEEP_PATIENCE=5,
                  PERM_N_REPEATS=3, SHAP_BACKGROUND=20, SHAP_N_EXPLAIN=20,
                  SHAP_KERNEL_NSAMPLES=60)
SEED: int = CONFIG["SEED"]

NOTEBOOKS: Dict[str, str] = {
    "01_data": "SoftCom_01_Data_Validation.ipynb",
    "02_preprocessing": "SoftCom_02_Preprocessing.ipynb",
    "03_benchmark": "SoftCom_03_Benchmarking.ipynb",
    "04_xai": "SoftCom_04_Explainable_AI.ipynb",
    "05_figures": "SoftCom_05_Figures.ipynb",
    "06_export": "SoftCom_06_Export.ipynb",
}

# ----------------------------------------------------------------------------- Drive
LAUNCH_DIR: Path = Path.cwd()
try:
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            drive.mount("/content/drive")
        except Exception as exc:  # pragma: no cover - interactive auth
            raise RuntimeError(
                "Google Drive mount failed. Re-run this cell and accept the Drive "
                f"authorisation prompt. Original error: {exc}"
            ) from exc
    PROJECT_ROOT = Path("/content/drive/MyDrive") / CONFIG["DRIVE_PROJECT_DIR"]
else:
    PROJECT_ROOT = Path(os.environ.get("SOFTCOM_ROOT", LAUNCH_DIR / CONFIG["DRIVE_PROJECT_DIR"])).resolve()

DIRS: Dict[str, Path] = {
    name: PROJECT_ROOT / sub
    for name, sub in {
        "raw": "data/raw",
        "processed": "data/processed",
        "models": "models",
        "predictions": "predictions",
        "tables": "tables",
        "figures": "figures",
        "artifacts": "artifacts",
        "logs": "logs",
    }.items()
}
for _d in DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)  # so ./figures/ and ./tables/ resolve inside Google Drive
MANIFEST_PATH: Path = DIRS["artifacts"] / "manifest.json"


# ----------------------------------------------------------------------------- helpers
def set_global_seed(seed: int = SEED) -> None:
    """Seed python, numpy and (if installed) torch for reproducible runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch  # type: ignore

        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError:
        pass


def log(step: str, message: str) -> None:
    """Print a message and append it to logs/<step>.log on Drive."""
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    print(message)
    with open(DIRS["logs"] / f"{step}.log", "a", encoding="utf-8") as fh:
        fh.write(f"[{stamp}] {message}\n")


def sha256_file(path: Path) -> str:
    """Return the SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def rel(path: Path) -> str:
    """Path relative to the project root (as stored in the manifest)."""
    return str(Path(path).resolve().relative_to(PROJECT_ROOT))


def _atomic_write_json(obj: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, default=str)
    os.replace(tmp, path)


def save_json(obj: Any, path: Path) -> Path:
    """Write JSON atomically (numpy/pandas values are stringified if needed)."""
    _atomic_write_json(obj, path)
    return path


def load_manifest() -> Dict[str, Any]:
    """Load the cross-notebook run manifest (empty skeleton if absent)."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, encoding="utf-8") as fh:
            return json.load(fh)
    return {"project": "SoftCom myopia benchmark", "steps": {}}


def register_outputs(step: str, paths: Iterable[Path], extra: Optional[Dict[str, Any]] = None) -> None:
    """Record a finished notebook step and the SHA-256 of every file it wrote."""
    manifest = load_manifest()
    files = {}
    for p in paths:
        p = Path(p)
        if not p.exists():
            raise FileNotFoundError(f"[{step}] expected output was not written: {p}")
        files[rel(p)] = sha256_file(p)
    manifest["steps"][step] = {
        "notebook": NOTEBOOKS[step],
        "status": "ok",
        "finished_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "fast_mode": CONFIG["FAST_MODE"],
        "outputs": files,
        "extra": extra or {},
    }
    _atomic_write_json(manifest, MANIFEST_PATH)
    print(f"Manifest updated: step '{step}' -> {len(files)} files registered.")


def require_upstream(step: str, needed: Sequence[str]) -> None:
    """Fail loudly unless `step` ran successfully and its files are unchanged.

    `needed` are project-relative paths that this notebook will read.
    """
    manifest = load_manifest()
    info = manifest["steps"].get(step)
    if info is None or info.get("status") != "ok":
        raise RuntimeError(
            f"Upstream step '{step}' has not completed. Open and 'Run all' "
            f"{NOTEBOOKS[step]} first (results folder: {PROJECT_ROOT})."
        )
    for r in needed:
        p = PROJECT_ROOT / r
        if not p.exists():
            raise FileNotFoundError(f"Missing upstream file {r}; re-run {NOTEBOOKS[step]}.")
        expected = info["outputs"].get(r)
        if expected is None:
            raise RuntimeError(f"{r} is not registered by {NOTEBOOKS[step]}; re-run it.")
        if sha256_file(p) != expected:
            raise RuntimeError(
                f"{r} changed after {NOTEBOOKS[step]} finished (hash drift). "
                f"Re-run {NOTEBOOKS[step]} and the notebooks after it."
            )
    if info.get("fast_mode") != CONFIG["FAST_MODE"]:
        print(f"WARNING: {step} ran with FAST_MODE={info.get('fast_mode')}, this notebook uses "
              f"FAST_MODE={CONFIG['FAST_MODE']}.")
    print(f"Upstream OK: {step} ({info['finished_at']}), {len(needed)} files verified.")


sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
                     "legend.fontsize": 10, "savefig.bbox": "tight"})


def save_figure(fig: plt.Figure, name: str, footnote: bool = True) -> List[Path]:
    """Save `fig` as 300-DPI PNG and SVG in ./figures/ (on Drive) and close it."""
    if footnote:  # place below everything already drawn (rotated tick labels included)
        bottom = fig.get_tightbbox(fig.canvas.get_renderer()).y0 / fig.get_figheight()
        fig.text(0.995, min(bottom, 0.0) - 0.01, CONFIG["COHORT_LABEL"], ha="right", va="top", fontsize=8, color="0.45")
    out = []
    try:
        for ext in ("png", "svg"):
            p = DIRS["figures"] / f"{name}.{ext}"
            fig.savefig(p, dpi=300, bbox_inches="tight")
            out.append(p)
    except OSError as exc:
        raise RuntimeError(f"Could not write figure {name} to {DIRS['figures']}: {exc}") from exc
    finally:
        plt.close(fig)
    print(f"Saved figure: figures/{name}.png + .svg")
    return out


def show_saved(name: str) -> None:
    """Display a saved PNG inline (Colab/Jupyter only)."""
    try:
        from IPython.display import Image, display  # type: ignore

        if "ipykernel" in sys.modules:
            display(Image(filename=str(DIRS["figures"] / f"{name}.png"), width=720))
    except ImportError:
        pass


set_global_seed(SEED)
print(f"Colab: {IN_COLAB} | results folder: {PROJECT_ROOT} | FAST_MODE={CONFIG['FAST_MODE']}")

Mounted at /content/drive
Colab: True | results folder: /content/drive/MyDrive/SoftCom_Myopia | FAST_MODE=False


## 1.1 Ingest the dataset into Google Drive
Put `full_myopia_hybrid_dataset_N1500.csv` in `MyDrive/SoftCom_Myopia/data/raw/`. If it is not
there, the cell copies it from the Colab working folder or opens an upload dialog, then saves it to Drive.

In [ ]:
STEP = "01_data"
RAW_PATH: Path = DIRS["raw"] / CONFIG["DATA_FILENAME"]


def ensure_raw_dataset(dest: Path) -> Path:
    """Make sure the raw CSV exists on Drive; copy or upload it if needed."""
    if dest.exists():
        return dest
    for cand in (LAUNCH_DIR / dest.name, Path("/content") / dest.name, Path("/content/sample_data") / dest.name):
        if cand.exists():
            shutil.copy2(cand, dest)
            log(STEP, f"Copied {cand} -> {rel(dest)}")
            return dest
    if IN_COLAB:
        from google.colab import files  # type: ignore

        print(f"Upload {dest.name} (it will be stored in Drive at {dest}).")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file uploaded.")
        name = dest.name if dest.name in uploaded else next(iter(uploaded))
        dest.write_bytes(uploaded[name])
        log(STEP, f"Uploaded {name} -> {rel(dest)}")
        return dest
    raise FileNotFoundError(f"Dataset not found. Place {dest.name} in {dest.parent} and re-run.")


RAW_PATH = ensure_raw_dataset(RAW_PATH)
try:
    df_raw = pd.read_csv(RAW_PATH)
except Exception as exc:
    raise RuntimeError(f"Could not parse {RAW_PATH}: {exc}. Check the file is the original CSV.") from exc
log(STEP, f"Loaded {rel(RAW_PATH)}: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols | sha256={sha256_file(RAW_PATH)[:12]}")

Upload full_myopia_hybrid_dataset_N1500.csv (it will be stored in Drive at /content/drive/MyDrive/SoftCom_Myopia/data/raw/full_myopia_hybrid_dataset_N1500.csv).


Saving full_myopia_hybrid_dataset_N1500.csv to full_myopia_hybrid_dataset_N1500.csv
Uploaded full_myopia_hybrid_dataset_N1500.csv -> data/raw/full_myopia_hybrid_dataset_N1500.csv
Loaded data/raw/full_myopia_hybrid_dataset_N1500.csv: 1500 rows x 12 cols | sha256=2e5ffbd144b9


## 1.2 Schema assertions (fail loudly on drift)

In [ ]:
EXPECTED_SCHEMA: Dict[str, str] = {
    "age": "int", "gender": "int", "parental_myopia_count": "int", "screen_hours_daily": "float",
    "continuous_reading_min": "float", "outdoor_hours_daily": "float", "diet_sugary_freq": "int",
    "axial_length_mm": "float", "corneal_power_d": "float", "anterior_chamber_depth": "float",
    "spherical_equivalent_d": "float", "is_myopic": "int",
}
# Plausible clinical / questionnaire domains (hard limits)
VALUE_DOMAINS: Dict[str, Tuple[float, float]] = {
    "age": (5, 25), "screen_hours_daily": (0, 24), "continuous_reading_min": (0, 300),
    "outdoor_hours_daily": (0, 16), "axial_length_mm": (18.0, 32.0), "corneal_power_d": (36.0, 50.0),
    "anterior_chamber_depth": (1.8, 5.0), "spherical_equivalent_d": (-20.0, 10.0),
}
ALLOWED_LEVELS: Dict[str, List[int]] = {
    "gender": [0, 1], "parental_myopia_count": [0, 1, 2], "diet_sugary_freq": [0, 1], "is_myopic": [0, 1],
}


def assert_schema(df: pd.DataFrame) -> None:
    """Raise if columns, dtypes, row count or categorical levels drift from the contract."""
    missing = sorted(set(EXPECTED_SCHEMA) - set(df.columns))
    extra = sorted(set(df.columns) - set(EXPECTED_SCHEMA))
    if missing or extra:
        raise ValueError(f"Schema drift. Missing columns: {missing}; unexpected columns: {extra}")
    for col, kind in EXPECTED_SCHEMA.items():
        ok = pd.api.types.is_integer_dtype(df[col]) if kind == "int" else pd.api.types.is_numeric_dtype(df[col])
        if not ok:
            raise TypeError(f"Column {col} should be {kind}, found {df[col].dtype}")
    if len(df) != CONFIG["EXPECTED_N_ROWS"]:
        raise ValueError(f"Expected {CONFIG['EXPECTED_N_ROWS']} rows, found {len(df)}. "
                         "Update CONFIG['EXPECTED_N_ROWS'] only if the dataset was intentionally changed.")
    for col, levels in ALLOWED_LEVELS.items():
        bad = sorted(set(df[col].dropna().unique()) - set(levels))
        if bad:
            raise ValueError(f"Column {col} has unexpected levels {bad} (allowed {levels})")
    print(f"Schema OK: {len(df)} rows, {df.shape[1]} columns, dtypes and levels as expected.")


assert_schema(df_raw)
df = df_raw.copy()
n_missing = int(df.isna().sum().sum())
n_dupes = int(df.duplicated().sum())
neg_zero = int((np.signbit(df.select_dtypes("float")) & (df.select_dtypes("float") == 0)).sum().sum())
df[df.select_dtypes("float").columns] = df.select_dtypes("float") + 0.0  # normalise -0.0 to 0.0
log(STEP, f"Missing cells: {n_missing} | duplicate rows: {n_dupes} (kept) | -0.0 values normalised: {neg_zero}")
log(STEP, "Rows dropped: 0")

Schema OK: 1500 rows, 12 columns, dtypes and levels as expected.
Missing cells: 0 | duplicate rows: 0 (kept) | -0.0 values normalised: 19
Rows dropped: 0


## 1.3 Validation report
Hard checks stop the pipeline. Soft checks are recorded as WARN and carried into `results.json`.

In [ ]:
TARGET = CONFIG["TARGET"]
KERATOMETRIC_FACTOR = 337.5  # corneal radius (mm) = 337.5 / keratometry (D), standard keratometric index 1.3375
report: List[Dict[str, Any]] = []


def add_check(name: str, value: Any, threshold: str, passed: Optional[bool], severity: str, note: str = "") -> None:
    """Append a check; passed=None marks INFO/SKIPPED rows."""
    if passed is None:
        status = "INFO"
    else:
        status = "PASS" if passed else ("FAIL" if severity == "hard" else "WARN")
    report.append({"check": name, "value": value, "threshold": threshold, "status": status,
                   "severity": severity, "note": note})


add_check("schema (columns, dtypes, levels)", "ok", "exact match", True, "hard")
add_check("row count", len(df), f"== {CONFIG['EXPECTED_N_ROWS']}", len(df) == CONFIG["EXPECTED_N_ROWS"], "hard")
add_check("missing cells", n_missing, "== 0 (imputed in-pipeline otherwise)", n_missing == 0, "soft")
add_check("duplicate rows", n_dupes, "== 0", n_dupes == 0, "soft")
for col, (lo, hi) in VALUE_DOMAINS.items():
    out = int(((df[col] < lo) | (df[col] > hi)).sum())
    add_check(f"domain {col}", f"[{df[col].min():.2f}, {df[col].max():.2f}]", f"within [{lo}, {hi}]", out == 0, "hard",
              f"{out} rows outside")

se_label = (df["spherical_equivalent_d"] <= -0.5).astype(int)
agree_label = float((se_label == df[TARGET]).mean())
add_check("label == (SE <= -0.50 D)", round(agree_label, 4), ">= 0.99", agree_label >= 0.99, "hard",
          "SE defines the label -> SE is excluded from model features (leakage)")
prev = float(df[TARGET].mean())
add_check("myopia prevalence", round(prev, 3), "0.10-0.90", 0.10 <= prev <= 0.90, "hard",
          "spec anchor ~0.47; moderate imbalance handled with class_weight='balanced'")

cr = KERATOMETRIC_FACTOR / df["corneal_power_d"]
al_cr = df["axial_length_mm"] / cr
agree_alcr = float(((al_cr > 3.0).astype(int) == df[TARGET]).mean())
add_check("AL/CR > 3 vs label agreement", round(agree_alcr, 3), ">= 0.85", agree_alcr >= 0.85, "soft",
          f"AL/CR median {al_cr.median():.3f}; biometry and refraction are weakly coupled in this cohort")

rho = df.assign(al_cr_ratio=al_cr).corr(method="spearman")
DIRECTION_CHECKS = [  # (a, b, expected sign, clinical rationale)
    ("axial_length_mm", "spherical_equivalent_d", -1, "longer eye -> more myopic SE"),
    ("al_cr_ratio", "spherical_equivalent_d", -1, "AL/CR is the strongest refraction proxy"),
    ("axial_length_mm", "anterior_chamber_depth", +1, "ocular growth co-scaling"),
    ("axial_length_mm", "corneal_power_d", -1, "emmetropisation: long eyes have flatter corneas"),
    ("age", "axial_length_mm", +1, "axial elongation with age"),
    ("parental_myopia_count", TARGET, +1, "heritability"),
    ("outdoor_hours_daily", TARGET, -1, "outdoor time is protective"),
    ("screen_hours_daily", TARGET, +1, "near work is a risk factor"),
]
for a, b, sign, why in DIRECTION_CHECKS:
    r = float(rho.loc[a, b])
    add_check(f"Spearman sign {a} ~ {b}", round(r, 3), "> 0" if sign > 0 else "< 0", np.sign(r) == sign, "soft", why)
r_al_acd = float(rho.loc["axial_length_mm", "anterior_chamber_depth"])
add_check("|rho(AL,ACD) - 0.50|", round(abs(r_al_acd - 0.5), 3), "<= 0.05", abs(r_al_acd - 0.5) <= 0.05, "soft",
          "spec anchor pair")
leaky = [c for c in df.columns if c != TARGET and abs(rho.loc[c, TARGET]) > 0.75]
add_check("near-deterministic predictors of label", ", ".join(leaky) or "none", "|rho| > 0.75 flagged", None,
          "info", "reviewed in notebook 02 (SE excluded; biometry kept - it is the clinical signal)")
add_check("TSTR/TRTR AUC gap", "n/a", "<= 0.05", None, "info",
          "SKIPPED: the provided file has no separate real/synthetic partition")

# Optional pilot comparison (KS for continuous, chi-square for categorical)
PILOT_PATH = DIRS["raw"] / CONFIG["PILOT_FILENAME"]
if PILOT_PATH.exists():
    from scipy.stats import chi2_contingency, ks_2samp

    pilot = pd.read_csv(PILOT_PATH)
    log(STEP, f"Pilot file found: {len(pilot)} rows (kept separate, comparison only)")
    for col in [c for c in EXPECTED_SCHEMA if c in pilot.columns and c != TARGET]:
        if col in ALLOWED_LEVELS:
            tab = pd.crosstab(np.r_[np.zeros(len(df)), np.ones(len(pilot))], np.r_[df[col], pilot[col]])
            p = chi2_contingency(tab)[1]
            add_check(f"chi2 main vs pilot: {col}", round(p, 4), "p >= 0.05", p >= 0.05, "soft")
        else:
            p = ks_2samp(df[col].dropna(), pilot[col].dropna()).pvalue
            add_check(f"KS main vs pilot: {col}", round(p, 4), "p >= 0.05", p >= 0.05, "soft")
else:
    add_check("pilot KS / chi-square", "n/a", "p >= 0.05", None, "info", f"no {CONFIG['PILOT_FILENAME']} in data/raw")

report_df = pd.DataFrame(report)
print(report_df[["check", "value", "threshold", "status"]].to_string(index=False))
n_fail = int((report_df.status == "FAIL").sum())
n_warn = int((report_df.status == "WARN").sum())
VALIDATION_STATUS = "FAIL" if n_fail else ("PASS with warnings" if n_warn else "PASS")
print(f"\nValidation: {VALIDATION_STATUS} | {int((report_df.status == 'PASS').sum())} pass, {n_warn} warn, {n_fail} fail")
if n_fail:
    report_df.to_csv(DIRS["tables"] / "table0_validation_report.csv", index=False)
    raise RuntimeError("Hard validation checks failed - see table0_validation_report.csv. Pipeline stopped.")

                                                 check                                   value                            threshold status
                      schema (columns, dtypes, levels)                                      ok                          exact match   PASS
                                             row count                                    1500                              == 1500   PASS
                                         missing cells                                       0 == 0 (imputed in-pipeline otherwise)   PASS
                                        duplicate rows                                       0                                 == 0   PASS
                                            domain age                           [6.00, 21.00]                       within [5, 25]   PASS
                             domain screen_hours_daily                           [0.50, 12.00]                       within [0, 24]   PASS
                         do

## 1.4 Table 1: cohort characteristics by myopia status

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

CATEGORICAL_COLS = ["gender", "parental_myopia_count", "diet_sugary_freq"]
CONTINUOUS_COLS = [c for c in EXPECTED_SCHEMA if c not in CATEGORICAL_COLS + [TARGET]]


def describe_by_group(data: pd.DataFrame) -> pd.DataFrame:
    """Median (IQR) + Mann-Whitney for continuous; n (%) + chi-square for categorical."""
    rows = []
    g0, g1 = data[data[TARGET] == 0], data[data[TARGET] == 1]
    for c in CONTINUOUS_COLS:
        f = lambda s: f"{s.median():.2f} ({s.quantile(.25):.2f}, {s.quantile(.75):.2f})"
        p = mannwhitneyu(g0[c], g1[c]).pvalue
        rows.append({"variable": c, "level": "P50 (P25, P75)", "overall": f(data[c]), "no_myopia": f(g0[c]),
                     "myopia": f(g1[c]), "p_value": p, "test": "Mann-Whitney U"})
    for c in CATEGORICAL_COLS:
        p = chi2_contingency(pd.crosstab(data[c], data[TARGET]))[1]
        for lvl in sorted(data[c].unique()):
            f = lambda s: f"{int((s == lvl).sum())} ({100 * (s == lvl).mean():.1f}%)"
            rows.append({"variable": c, "level": str(lvl), "overall": f(data[c]), "no_myopia": f(g0[c]),
                         "myopia": f(g1[c]), "p_value": p, "test": "chi-square"})
    out = pd.DataFrame(rows)
    out["p_value"] = out["p_value"].map(lambda v: "<0.001" if v < 0.001 else f"{v:.3f}")
    return out


table1 = describe_by_group(df)
print(f"n no myopia = {(df[TARGET] == 0).sum()}, n myopia = {(df[TARGET] == 1).sum()}")
print(table1.to_string(index=False))

n no myopia = 1032, n myopia = 468
              variable          level              overall            no_myopia               myopia p_value           test
                   age P50 (P25, P75) 14.00 (10.00, 17.00) 13.00 (10.00, 17.00) 16.00 (11.00, 18.00)  <0.001 Mann-Whitney U
    screen_hours_daily P50 (P25, P75)    3.00 (1.70, 4.50)    3.00 (1.40, 4.40)    3.30 (2.10, 5.60)  <0.001 Mann-Whitney U
continuous_reading_min P50 (P25, P75) 71.50 (46.00, 92.00) 70.00 (48.00, 90.00) 76.50 (43.00, 97.00)   0.094 Mann-Whitney U
   outdoor_hours_daily P50 (P25, P75)    0.80 (0.40, 1.50)    0.80 (0.40, 1.62)    0.60 (0.30, 1.12)  <0.001 Mann-Whitney U
       axial_length_mm P50 (P25, P75) 24.09 (23.47, 24.75) 23.72 (23.24, 24.14) 25.10 (24.75, 25.53)  <0.001 Mann-Whitney U
       corneal_power_d P50 (P25, P75) 43.22 (42.41, 44.01) 43.44 (42.62, 44.17) 42.83 (42.01, 43.59)  <0.001 Mann-Whitney U
anterior_chamber_depth P50 (P25, P75)    3.58 (3.42, 3.74)    3.52 (3.38, 3.68)    3.69 (3.57, 3.

## 1.5 Save to Drive and register in the manifest

In [ ]:
df_out = df.copy()
df_out.insert(0, "row_id", np.arange(len(df_out)))
df_out["source"] = CONFIG["SOURCE_LABEL"]
paths = {
    "cohort": DIRS["processed"] / "cohort_validated.csv",
    "report_csv": DIRS["tables"] / "table0_validation_report.csv",
    "table1": DIRS["tables"] / "table1_characteristics.csv",
    "report_json": DIRS["artifacts"] / "validation_report.json",
}
try:
    df_out.to_csv(paths["cohort"], index=False)
    report_df.to_csv(paths["report_csv"], index=False)
    table1.to_csv(paths["table1"], index=False)
    save_json({"status": VALIDATION_STATUS, "n_rows": len(df), "prevalence": prev,
               "raw_sha256": sha256_file(RAW_PATH), "checks": report}, paths["report_json"])
except OSError as exc:
    raise RuntimeError(f"Writing to Google Drive failed ({exc}). Check Drive is mounted and has free space.") from exc
register_outputs(STEP, list(paths.values()) + [RAW_PATH],
                 {"validation_status": VALIDATION_STATUS, "n_rows": len(df), "prevalence": prev})

Manifest updated: step '01_data' -> 5 files registered.


In [ ]:
print("=" * 70)
print(f"01 DONE | rows={len(df)} | prevalence={prev:.3f} | validation={VALIDATION_STATUS}")
print(f"Outputs in {PROJECT_ROOT}: data/processed/cohort_validated.csv, tables/table0+1, artifacts/validation_report.json")
print(f"NEXT -> open {NOTEBOOKS['02_preprocessing']} and Run all")

01 DONE | rows=1500 | prevalence=0.312 | validation=PASS with warnings
Outputs in /content/drive/MyDrive/SoftCom_Myopia: data/processed/cohort_validated.csv, tables/table0+1, artifacts/validation_report.json
NEXT -> open SoftCom_02_Preprocessing.ipynb and Run all
